# Week 8-A: Counterfactual Single-Passage Promotion Response Surface
## Draft-Aware Fidelity Routing (DAFiR) for Compressed RAG

This notebook measures the exact counterfactual marginal promotion utility $\Delta_i(q) = F_i(q) - F_0(q)$
for every candidate passage $i \in \{5, 6, 7, 8, 9, 10\}$ relative to the canonical semantic $k=4$ baseline ($4F + 6H$).

### Scientific Guarantees:
- Tests whether factual gains from ranks 5-10 can be unlocked without context bloat.
- Classifies each candidate into **HELP** (> +0.01), **NEUTRAL** ([-0.01, +0.01]), or **HURT** (< -0.01).
- Evaluates **Oracle-1**, **Oracle-Selective-1** ($\epsilon=0.01$), and **Greedy Oracle-2**.
- Strictly maintains paper-disjointness between FIT and INTERNAL-VAL.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
SARA_ROOT = REPO_ROOT / "Baselines" / "SARA-main"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import src
if hasattr(src, "__path__") and str(SARA_ROOT / "src") not in src.__path__:
    src.__path__.append(str(SARA_ROOT / "src"))

from src.model.loader import load_sara_for_eval
from src.week8.semantic import SemanticScorer
from src.week8.data import load_config, load_week8_split
from src.week8.promotion_surface import run_single_promotion_surface
from src.week8.promotion_oracle import (
    compute_oracle_1,
    compute_oracle_selective_1,
    compute_best_rank_distribution,
    plot_oracle_results,
)

print("Environment and modules loaded successfully.")


In [ ]:
# Load configuration and split data
cfg = load_config()
split = "internal_val"  # Change to 'fit' for full training surface
records, sfr_dict, manifest_df = load_week8_split(split, cfg)
print(f"Loaded {len(records)} queries for split '{split}'.")


In [ ]:
# Compute semantic similarity rankings
scorer = SemanticScorer()
semantic_scores_by_qid = {}
for r in records:
    qid = str(r["question_id"])
    q_text = r["question"]
    passages = [p["text"] for p in r.get("passages", [])[:10]]
    sims = scorer.score_query_passages(q_text, passages).cpu().numpy()
    semantic_scores_by_qid[qid] = sims

print(f"Computed semantic rankings for {len(semantic_scores_by_qid)} queries.")


In [ ]:
# Load SARA generator
device = "cuda" if torch.cuda.is_available() else "cpu"
sara_ckpt_path = REPO_ROOT / cfg["sara"]["checkpoint_path"]
sara_model, sara_tokenizer = load_sara_for_eval(str(sara_ckpt_path), device=device)
print(f"Loaded SARA generator on {device}.")


In [ ]:
# Run counterfactual single-passage promotion surface
cache_path = REPO_ROOT / "results/week8/surfaces" / f"promotion_surface_{split}.jsonl"
df_promotions = run_single_promotion_surface(
    sara_model=sara_model,
    sara_tokenizer=sara_tokenizer,
    records=records,
    semantic_scores_by_qid=semantic_scores_by_qid,
    cached_sfr_embeddings=sfr_dict,
    cache_path=cache_path,
    device=device,
)
print(f"Promotion surface completed with {len(df_promotions)} candidate evaluations.")


In [ ]:
# Compute Oracle-1 and Oracle-Selective-1
df_o1, o1_summary = compute_oracle_1(df_promotions)
df_sel1, sel1_summary = compute_oracle_selective_1(df_promotions, epsilon=0.01)
df_ranks = compute_best_rank_distribution(df_o1)

print("=== Oracle-1 Summary ===")
for k, v in o1_summary.items():
    print(f"  {k}: {v}")

print("\n=== Oracle-Selective-1 Summary ===")
for k, v in sel1_summary.items():
    print(f"  {k}: {v}")


In [ ]:
# Plot distribution and quality-context frontiers
out_figures = REPO_ROOT / "results/week8/figures"
plot_oracle_results(df_o1, df_sel1, out_figures)
print(f"Saved figures to {out_figures}")
